# M2 · Segmentación de instancias de vehículos (YOLO11s-seg)
**IAM-811 · Laboratorio Final — Sistema Inteligente de Control de Peaje**

Clases internas (definen la tarifa): **`auto` · `minibus` · `bus` · `camion`** y opcional **`moto`**.
COCO no tiene *minibús* (el transporte público paceño), por eso se entrena un modelo propio con máscaras
etiquetadas en **Roboflow** (proyecto *Instance Segmentation*).

**Salida:** `m2_vehiculos_seg.pt` → copiar a `inferencia/pesos/` del Pod de RunPod.
**Métricas:** mAP50 y mAP50-95 de caja y de máscara (global y por clase) + matriz de confusión.

## 1. Instalación

In [ ]:
!nvidia-smi -L || echo "ACTIVE LA GPU T4: Entorno de ejecución > Cambiar tipo de entorno"
!pip -q install ultralytics==8.4.218 roboflow
import os, json, glob, shutil, yaml, torch, pandas as pd, matplotlib.pyplot as plt, cv2, numpy as np
from ultralytics import YOLO
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO')

## 2. Dataset desde Roboflow
* En Roboflow: *Create Project → Instance Segmentation*, clases **exactamente** `auto`, `minibus`, `bus`,
  `camion` (y `moto` si tiene datos). Recomendado ≥ 150 instancias por clase, fotos y frames de video de La Paz.
* Use **Smart Polygon (SAM)** para acelerar las máscaras. *Generate* con augmentations (flip, brillo ±25 %).
* API key: **Colab → 🔑 Secretos → `ROBOFLOW_API_KEY`** (nunca la escriba en el notebook).

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

RF_WORKSPACE = 'tu-workspace'          # <- cambie
RF_PROYECTO  = 'vehiculos-bolivia-seg' # <- cambie
RF_VERSION   = 1

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
ds = rf.workspace(RF_WORKSPACE).project(RF_PROYECTO).version(RF_VERSION).download('yolov8')
DATA_YAML = f'{ds.location}/data.yaml'
cfg = yaml.safe_load(open(DATA_YAML)); print(cfg['names'])

### 2.1 Normalizar nombres de clase (por si en Roboflow se usaron otros nombres)

In [ ]:
NORMALIZAR = {'car': 'auto', 'auto particular': 'auto', 'vagoneta': 'auto', 'van': 'minibus', 'minibús': 'minibus',
              'micro': 'minibus', 'truck': 'camion', 'camión': 'camion', 'motorcycle': 'moto', 'motocicleta': 'moto'}
nombres = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]
cfg['names'] = [NORMALIZAR.get(n.lower(), n.lower()) for n in nombres]
yaml.safe_dump(cfg, open(DATA_YAML, 'w'), allow_unicode=True)
assert {'auto', 'minibus', 'bus', 'camion'} <= set(cfg['names']), 'Faltan clases mínimas: ' + str(cfg['names'])
print('Clases finales:', cfg['names'])

# Conteo de instancias por clase y split
conteo = []
for split in ('train', 'valid', 'test'):
    for f in glob.glob(f'{ds.location}/{split}/labels/*.txt'):
        for linea in open(f):
            if linea.strip():
                conteo.append({'split': split, 'clase': cfg['names'][int(linea.split()[0])]})
pd.DataFrame(conteo).value_counts().unstack(fill_value=0)

## 3. Línea base: modelo COCO preentrenado (lo que usa el servidor en modo *fallback*)

In [ ]:
base = YOLO('yolo11s-seg.pt')
img_val = sorted(glob.glob(f'{ds.location}/valid/images/*'))[:4]
fig, ax = plt.subplots(1, len(img_val), figsize=(5 * len(img_val), 5))
for a, p in zip(np.atleast_1d(ax), img_val):
    a.imshow(base.predict(p, conf=0.35, verbose=False)[0].plot()[:, :, ::-1]); a.axis('off')
plt.suptitle('COCO: no distingue minibús (lo etiqueta car/bus/truck)'); plt.show()

## 4. Entrenamiento (T4 ≈ 25–40 min para 80 épocas)

In [ ]:
modelo = YOLO('yolo11s-seg.pt')
modelo.train(data=DATA_YAML, epochs=80, imgsz=640, batch=16, patience=20, cos_lr=True,
             project='/content/runs/segment', name='m2', exist_ok=True, device=0, plots=True)

## 5. Métricas: mAP50 y mAP50-95 (caja y máscara)

In [ ]:
best = YOLO('/content/runs/segment/m2/weights/best.pt')
val = best.val(data=DATA_YAML, split='valid', plots=True, project='/content/runs/segment', name='m2_val', exist_ok=True)
resumen = pd.DataFrame([{
    'mAP50 (box)': val.box.map50, 'mAP50-95 (box)': val.box.map,
    'mAP50 (mask)': val.seg.map50, 'mAP50-95 (mask)': val.seg.map,
    'precision (mask)': val.seg.mp, 'recall (mask)': val.seg.mr}]).round(4)
por_clase = pd.DataFrame({
    'clase': [best.names[i] for i in val.box.ap_class_index],
    'mAP50 box': val.box.ap50, 'mAP50-95 box': val.box.ap,
    'mAP50 mask': val.seg.ap50, 'mAP50-95 mask': val.seg.ap}).round(4)
json.dump({'global': resumen.iloc[0].to_dict(), 'por_clase': por_clase.to_dict('records')},
          open('/content/m2_metricas.json', 'w'), indent=2)
display(resumen); display(por_clase)

## 6. Matriz de confusión y curvas

In [ ]:
from IPython.display import Image, display
for f in ['confusion_matrix_normalized.png', 'MaskPR_curve.png', 'BoxPR_curve.png']:
    p = f'/content/runs/segment/m2_val/{f}'
    if os.path.exists(p): display(Image(p, width=700))
display(Image('/content/runs/segment/m2/results.png', width=1000))

## 7. Predicciones de validación

In [ ]:
imgs = sorted(glob.glob(f'{ds.location}/valid/images/*'))[:6]
fig, ax = plt.subplots(2, 3, figsize=(18, 10)); ax = ax.ravel()
for a, p in zip(ax, imgs):
    r = best.predict(p, conf=0.35, verbose=False)[0]
    a.imshow(r.plot()[:, :, ::-1]); a.axis('off')
    a.set_title(', '.join(f'{best.names[int(c)]} {s:.2f}' for c, s in zip(r.boxes.cls, r.boxes.conf)))
plt.tight_layout(); plt.show()

## 8. Exportar `m2_vehiculos_seg.pt` (y copia en Google Drive)

In [ ]:
shutil.copy('/content/runs/segment/m2/weights/best.pt', '/content/m2_vehiculos_seg.pt')
print(YOLO('/content/m2_vehiculos_seg.pt').names)   # el servidor normaliza estos nombres
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/peaje-ia/pesos', exist_ok=True)
    shutil.copy('/content/m2_vehiculos_seg.pt', '/content/drive/MyDrive/peaje-ia/pesos/')
    shutil.copy('/content/m2_metricas.json', '/content/drive/MyDrive/peaje-ia/')
except Exception as e:
    print('Drive no montado:', e)
from google.colab import files
files.download('/content/m2_vehiculos_seg.pt'); files.download('/content/m2_metricas.json')